# LITHOLOGY PREDICTOR

Enter five log readings, get the estimated rock type with probabilities.

The model is trained on `force2020_lithology.csv`, whose labels came from the clustering in
`feature_engineering_final.ipynb`. So it learns to reproduce that interpretation — a high score means it
matches the clustering, **not** that the geology is verified.

Training uses the **cleaned but unsmoothed** readings, because a single typed-in reading has no neighbours to
smooth with.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

data = pd.read_csv("force2020_lithology.csv")
curves = ["RHOB", "GR", "NPHI", "PEF", "DTC"]

labelled = data.dropna(subset=["lithology"] + curves).copy()
print("Labelled depths:", len(labelled),
      "| dropped for a blank reading:", int(data["lithology"].notna().sum() - len(labelled)))
labelled["lithology"].value_counts()

In [ ]:
# split by 50 m depth blocks: neighbouring depths are 0.152 m apart and nearly identical,
# so a shuffled split would put near-duplicates in both train and test and inflate the score
labelled["block"] = (labelled["DEPTH_MD"] // 50).astype(int)

X = labelled[curves]
y = labelled["lithology"]
groups = labelled["block"]

print("Depth blocks:", groups.nunique())

In [ ]:
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
blocked = GroupKFold(n_splits=5)

results = []
for name, model in [("Random Forest", forest), ("Logistic Regression", logreg)]:
    honest = cross_val_score(model, X, y, cv=blocked, groups=groups, n_jobs=-1)
    leaky = cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=42), n_jobs=-1)
    results.append({"model": name,
                    "blocked CV (honest)": round(honest.mean(), 3),
                    "worst block fold": round(honest.min(), 3),
                    "shuffled split (leaky)": round(leaky.mean(), 3)})

pd.DataFrame(results).set_index("model")

In [ ]:
predicted = cross_val_predict(forest, X, y, cv=blocked, groups=groups, n_jobs=-1)
print(classification_report(y, predicted, digits=3))

names = sorted(y.unique())
matrix = pd.DataFrame(confusion_matrix(y, predicted, labels=names), index=names, columns=names)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", linewidths=0.5, linecolor="white", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("From the clustering")
ax.set_title("Where the predictor disagrees with the clustering")
plt.tight_layout()
plt.show()

In [ ]:
forest.fit(X, y)

importance = pd.Series(forest.feature_importances_, index=curves).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(importance.index, importance.values, color="#2a78d6")
ax.set_xlabel("Importance")
ax.set_title("Which curves the model relies on")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

importance.sort_values(ascending=False).round(3)

In [ ]:
def predict_lithology(rhob, gr, nphi, pef, dtc):
    reading = pd.DataFrame([[rhob, gr, nphi, pef, dtc]], columns=curves)
    probabilities = pd.Series(forest.predict_proba(reading)[0], index=forest.classes_)
    return probabilities.sort_values(ascending=False).round(3)


examples = {
    "dense, clean, fast": (2.55, 16, 0.17, 4.8, 70),
    "light, clayey, slow": (2.01, 68, 0.50, 2.6, 146),
    "dense, clayey": (2.48, 96, 0.31, 4.4, 88),
    "halfway between": (2.25, 55, 0.35, 3.8, 110),
}

for description, reading in examples.items():
    top = predict_lithology(*reading)
    print(f"{description:22} RHOB {reading[0]}, GR {reading[1]}, NPHI {reading[2]}, PEF {reading[3]}, DTC {reading[4]}")
    print("   ", " | ".join(f"{name}: {value:.0%}" for name, value in top.items()), "\n")

In [ ]:
joblib.dump({"model": forest, "curves": curves, "classes": list(forest.classes_)}, "lithology_model.joblib")

# how to use it elsewhere:
loaded = joblib.load("lithology_model.joblib")
reading = pd.DataFrame([[2.55, 16, 0.17, 4.8, 70]], columns=loaded["curves"])
print(dict(zip(loaded["classes"], loaded["model"].predict_proba(reading)[0].round(3))))

### Result

**Accuracy 97.7%** on depth-blocked cross-validation (worst block 93.6%), meaning the model reproduces the
clustering it was trained on. Random Forest and Logistic Regression score the same, which says the three rock
types are separable by simple boundaries in the five curves.

**Why the split matters:** with a shuffled split the same model scores 98.9%, because readings 0.152 m apart
are nearly identical and end up in both train and test. Splitting by 50 m depth blocks removes that.

**Where it disagrees with the clustering:** almost entirely at layer boundaries — carbonate confused with soft
shale (100 depths) and with compacted shale (65). Those are transition depths where the readings genuinely sit
between two rocks. Soft shale, the thickest unit, is 99% right.

**Which curves it uses:** DTC (0.34), NPHI (0.30) and GR (0.21) carry most of the decision, RHOB 0.13, PEF
0.02. PEF contributes little because it was the curve most affected by borehole problems, so cleaning removed
much of its signal.

**What this model is:** a fast way to apply the interpretation to new readings. It cannot be more correct than
the clustering it learned from, and it only knows the three rock types present in this well — give it a
sandstone or a coal and it will still answer with one of these three.